In [ ]:
import json
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import polars as pl
import powerlaw
import seaborn as sns
from scipy.stats import gaussian_kde, pareto

from graphsim import paths


In [ ]:
sns.set_theme()

# Load dataset
production_capabilities_df = pl.read_csv(paths.DATA_DIR / "ProductionCapabilities.csv")
products_df = pl.read_csv(paths.DATA_DIR / "Products.csv")
special_products_df = pl.read_csv(paths.DATA_DIR / "SpecialProducts.csv")
special_production_capabilities_df = pl.read_csv(paths.DATA_DIR / "SpecialProductionCapabilities.csv")
supply_relations_df = pl.read_csv(paths.DATA_DIR / "SupplyRelations.csv")
supply_relation_details_df = pl.read_csv(paths.DATA_DIR / "SupplyRelationDetails.csv")
firms_df = pl.read_csv(paths.DATA_DIR / "Firms.csv", infer_schema_length=10000)
countries_df = pl.read_csv(paths.DATA_DIR / "GeoNations.csv")

In [ ]:
# Create networkx supply network
supply_net: nx.DiGraph = nx.from_pandas_edgelist(
    supply_relations_df,
    source="source_firm_id",
    target="target_firm_id",
    create_using=nx.DiGraph,
)

# Add isolated nodes
supply_net.add_nodes_from(firms_df["firm_id"])

# Add node attributes
node_attr = firms_df.rows_by_key("firm_id", named=True, unique=True)
nx.set_node_attributes(supply_net, node_attr)

In [ ]:
# Compute degrees
node_degree = dict(supply_net.degree())
node_in_degree = dict(supply_net.in_degree())
node_out_degree = dict(supply_net.out_degree())

# Compute centralities
# betweenness_centrality = nx.betweenness_centrality(supply_net) # Computationally expensive
closeness_centrality = nx.closeness_centrality(supply_net)
eigenvector_centrality = nx.eigenvector_centrality(supply_net)
pagerank = nx.pagerank(supply_net)

# Compute clustering coefficient
clustering_coefficient = nx.clustering(supply_net)

# Compute tiers
node_tier_dict = defaultdict(list)
reverse_net = supply_net.reverse(copy=True)
sinks = [node for node in supply_net.nodes if supply_net.out_degree(node) == 0]

# BFS from each sink node
for sink in sinks:
    # Perform BFS to find all reachable nodes
    for tier, layer in enumerate(nx.bfs_layers(reverse_net, sink)):
        for firm in layer:
            node_tier_dict[firm].append(tier)

# Calculate the median tier for each node
for node, tiers in node_tier_dict.items():
    tier = int(np.median(tiers))

    # Set tier to -1 if the node is isolated
    if node_degree[node] == 0:
        tier = -1

    node_tier_dict[node] = tier

In [ ]:
data = []

for node, attr in supply_net.nodes(data=True):
    record = {
        "firm_id": node,
        "degree": node_degree[node],
        "in_degree": node_in_degree[node],
        "out_degree": node_out_degree[node],
        "closeness_centrality": closeness_centrality[node],
        "eigenvector_centrality": eigenvector_centrality[node],
        "pagerank": pagerank[node],
        "clustering_coefficient": clustering_coefficient[node],
        "tier": node_tier_dict.get(node, -1),
        "nation_id": attr.get("nation_id", None),
    }
    data.append(record)

# Create dataframe
node_df = pl.DataFrame(data)
node_df = node_df.join(countries_df.select(["nation_id", "nation_name"]), left_on="nation_id", right_on="nation_id", how="left")

In [ ]:
def compute_ccdf(df: pl.DataFrame, col_name: str) -> pl.DataFrame:
    """Compute the CCDF for a discrete column `col_name` in `df`.

    Returns a DataFrame with columns:
       col_name     - the distinct values of that column,
       "count"      - how many rows had exactly that value,
       "cum_count"  - how many rows have value >= that value,
       "ccdf"       - cum_count / N (fraction with value >= that value).

    """
    # 2a) Total number of rows (N)
    total = df.height

    # 2b) Group by the column, count how many times each value appears
    freq = (
        df
        .group_by(col_name)
        .agg([
            pl.len().alias("count"),
        ])
        # At this point `freq` has columns: [ col_name, "count" ].
    )

    # 2c) Sort by `col_name` *descending*, so that we can cumulate from largest → smallest:
    freq_desc = freq.sort(col_name, descending=True)

    # 2d) Add a running sum ("cum_count") over the descending-sorted counts
    freq_desc = freq_desc.with_columns([
        pl.col("count").cum_sum().alias("cum_count"),
    ])

    # 2e) Now compute ccdf = cum_count / total
    freq_desc = freq_desc.with_columns([
        (pl.col("cum_count") / total).alias("ccdf"),
    ])

    # 2f) (Optional) If you want to see the results in ascending order of `col_name`,
    #     simply sort again by `col_name` ascending:
    return freq_desc.sort(col_name, descending=True)

In [ ]:
def fit_power_law(df: pl.DataFrame, col_name: str):
    """Fit a power law distribution to the data in `df[col_name]`."""
    fit = powerlaw.Fit(df[col_name], estimate_discrete=True)
    #R, p = fit.distribution_compare("power_law", "exponential")
    #print(f"R: {R}, p-value: {p}")
    #print(fit.alpha, fit.sigma, fit.xmin)

    # Positive R value shows that powerlaw is a much better fit than exponential
    # Better fit is statistically significant if p < 0.05

    # 10000 samples from powerlaw distribution
    b = fit.alpha - 1

    samples_powerlaw = pareto.rvs(b, size=1000000)
    powerlaw_df = pl.DataFrame({col_name: samples_powerlaw})
    return compute_ccdf(powerlaw_df, col_name)

# Network characteristics

The large number of isolates indicates that still major parts of the network are missing, especially edges.

In [ ]:
num_nodes = firms_df.select(pl.col("firm_id")).height
nodes_with_products = pl.concat([
    production_capabilities_df.select(pl.col("firm_id")),
    special_production_capabilities_df.select(pl.col("firm_id")),
]).unique()
num_nodes_with_products = nodes_with_products.height
oems = firms_df.filter(pl.col("is_oem") == True).select(pl.col("firm_id"))
num_oems = oems.height
nodes_without_products = firms_df.filter(~pl.col("firm_id").is_in(pl.concat([nodes_with_products, oems])["firm_id"])).select(pl.col("firm_id"))
num_nodes_without_products = nodes_without_products.height
num_isolated_nodes = nx.number_of_isolates(supply_net)

num_edges = supply_relations_df.select(pl.col("relation_id")).n_unique()
num_edges_with_products = supply_relation_details_df.select(pl.col("relation_id")).n_unique()
num_edges_without_products = num_edges - num_edges_with_products

num_products = products_df.select(pl.col("product_id")).height
num_special_products = special_products_df.select(pl.col("special_product_id")).height

# Nodes with only special products
special_nodes = special_production_capabilities_df.select(pl.col("firm_id")).unique()
regular_nodes = production_capabilities_df.select(pl.col("firm_id")).unique()
nodes_with_only_special_products = special_nodes.join(regular_nodes, on="firm_id", how="anti")

# Print summary statistics
print(f"Number of nodes: {num_nodes}")
print(f"Number of OEMs: {num_oems}") # OEMs do not have products
print(f"Number of nodes with products: {num_nodes_with_products}")
print(f"Number of nodes without products: {num_nodes_without_products}")
print(f"Number of isolated nodes: {num_isolated_nodes}")
print(f"Number of edges: {num_edges}")
print(f"Number of edges with products: {num_edges_with_products}")
print(f"Number of edges without products: {num_edges_without_products}")
print(f"Number of products: {num_products}")
print(f"Number of special products: {num_special_products}")
print(f"Number of nodes with only special products: {nodes_with_only_special_products.height}")


In [ ]:
nodes_in_network_src = nodes_with_only_special_products.join(supply_relations_df, left_on="firm_id", right_on="source_firm_id", how="anti")
nodes_in_network_tgt = nodes_with_only_special_products.join(supply_relations_df, left_on="firm_id", right_on="target_firm_id", how="anti")
nodes_in_network = pl.concat([nodes_in_network_src, nodes_in_network_tgt]).unique()

print(f"Number of nodes with only special products in network: {nodes_in_network.height}")

# Degree distribution

In [ ]:
# Filter isolated nodes (degree == 0)
node_df2 = node_df.filter(pl.col("degree") > 0)

In [ ]:
# Get high-degree nodes (top 10%)
high_degree_nodes = node_df2.sort("degree", descending=True).head(int(node_df2.height * 0.1))

# Print range of high-degree nodes
print(
    "Range of high-degree nodes: ",
    high_degree_nodes["degree"].min(),
    high_degree_nodes["degree"].max(),
)

In [ ]:
ccdf_degree     = compute_ccdf(node_df2, "degree")
ccdf_in_degree  = compute_ccdf(node_df2, "in_degree")
ccdf_out_degree = compute_ccdf(node_df2, "out_degree")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

# Total degree distribution
sns.scatterplot(x=ccdf_degree["degree"], y=ccdf_degree["ccdf"], ax=ax1, s=100, edgecolor="none")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Degree")
ax1.set_ylabel("CCDF")
ax1.set_title("Total degree distribution")

# In-degree distribution
sns.scatterplot(x=ccdf_in_degree["in_degree"], y=ccdf_in_degree["ccdf"], ax=ax2, s=100, edgecolor="none")
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel("In-degree")
ax2.set_ylabel("CCDF")
ax2.set_title("In-degree distribution")

# Out-degree distribution
sns.scatterplot(x=ccdf_out_degree["out_degree"], y=ccdf_out_degree["ccdf"], ax=ax3, s=100, edgecolor="none")
ax3.set_xscale("log")
ax3.set_yscale("log")
ax3.set_xlabel("Out-degree")
ax3.set_ylabel("CCDF")
ax3.set_title("Out-degree distribution")

plt.tight_layout()
plt.show()

## Power-law fit and goodness-of-fit

We fit a discrete power law $P(k) \propto k^{-\alpha}$ to each degree series (total, in, out) using the maximum-likelihood estimator of `powerlaw` (Clauset, Shalizi & Newman 2009; Alstott et al. 2014). The fit is restricted to $k \geq x_\text{min}$, where $x_\text{min}$ is chosen to minimise the Kolmogorov–Smirnov distance between empirical and fitted CCDF. To assess whether a power law is the best heavy-tailed description of the data, we compare it against two alternatives via the normalised log-likelihood-ratio test: an exponential distribution (light tail) and a log-normal distribution (heavy tail with a different shape). $R > 0$ favours the power law; the associated $p$-value tests significance of the preference.

In [ ]:
# Fit discrete power-law and compare against exponential / log-normal
import warnings
warnings.filterwarnings("ignore")  # powerlaw produces verbose RuntimeWarnings on KS tail evaluation

degree_series = {
    "total":  np.array([d for _, d in supply_net.degree()     if d > 0]),
    "in":     np.array([d for _, d in supply_net.in_degree()  if d > 0]),
    "out":    np.array([d for _, d in supply_net.out_degree() if d > 0]),
}

powerlaw_fits = {}
print(f"{'Series':<8} {'n>0':>8} {'alpha':>7} {'xmin':>6} {'sigma':>8}  {'vs Exponential':>24}  {'vs Log-normal':>24}")
print("-" * 100)
for label, data in degree_series.items():
    fit = powerlaw.Fit(data, estimate_discrete=True, verbose=False)
    R_exp, p_exp = fit.distribution_compare("power_law", "exponential", normalized_ratio=True)
    R_ln,  p_ln  = fit.distribution_compare("power_law", "lognormal",   normalized_ratio=True)
    powerlaw_fits[label] = {
        "fit": fit, "alpha": fit.alpha, "xmin": fit.xmin, "sigma": fit.sigma,
        "R_exp": R_exp, "p_exp": p_exp, "R_ln": R_ln, "p_ln": p_ln,
    }
    print(f"{label:<8} {len(data):>8,} {fit.alpha:>7.3f} {int(fit.xmin):>6} {fit.sigma:>8.4f}  "
          f"R={R_exp:>+6.2f} p={p_exp:.3f}  R={R_ln:>+6.2f} p={p_ln:.3f}")

print()
print("Reading:")
print("  - alpha is the power-law exponent estimated on the tail (k >= xmin)")
print("  - sigma is the standard error on alpha")
print("  - R > 0 favours the power law over the alternative; p < 0.05 indicates the preference is significant.")
print("  - R < 0 indicates the alternative (exponential or log-normal) fits better than the power law.")

In [ ]:
# CCDF with power-law fit overlay for each degree series
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (label, info) in zip(axes, powerlaw_fits.items()):
    fit = info["fit"]
    fit.plot_ccdf(ax=ax, color="steelblue", marker="o", linestyle="None", markersize=4)
    fit.power_law.plot_ccdf(ax=ax, color="crimson", linestyle="--",
                            label=f"power-law fit\n$\\alpha = {info['alpha']:.2f}$, $x_\\mathrm{{min}} = {int(info['xmin'])}$")
    ax.set_xlabel(f"{label}-degree")
    ax.set_ylabel("CCDF")
    ax.set_title(f"{label.capitalize()} degree distribution")
    ax.legend(loc="lower left", fontsize=10)
plt.tight_layout()
plt.show()

# Distribution of products

In [ ]:
# Count products per company
products_per_company = production_capabilities_df.group_by("firm_id").agg(
    pl.count("product_id").alias("num_products"),
)
products_per_company = firms_df.join(products_per_company, how="left", on="firm_id").select(
    "firm_id", "num_products",
)
products_per_company = products_per_company.fill_null(0)
ccdf_products = compute_ccdf(products_per_company, "num_products")

# Count number of products
companies_per_product = production_capabilities_df.group_by("product_id").agg(
    pl.count("firm_id").alias("num_firms"),
)
# Sort
companies_per_product = companies_per_product.sort("num_firms")

ccdf_powerlaw = fit_power_law(products_per_company, "num_products")

In [ ]:
# Plot as scatter plot with log scale
sns.scatterplot(x=ccdf_products["num_products"], y=ccdf_products["ccdf"], s=30, edgecolor="none")
sns.scatterplot(x=ccdf_powerlaw["num_products"], y=ccdf_powerlaw["ccdf"], s=30, edgecolor="none")
plt.yscale("log")
plt.xscale("log")
plt.xlabel("Number of products")
plt.ylabel("CCDF")
plt.title("Number of products per company")

In [ ]:
# Count special products per company
special_products_per_company = special_production_capabilities_df.group_by("firm_id").agg(
    pl.count("special_product_id").alias("num_special_products"),
)
special_products_per_company = firms_df.join(special_products_per_company, how="left", on="firm_id").select(
    "firm_id", "num_special_products",
)
special_products_per_company = special_products_per_company.fill_null(0)
ccdf_special_products = compute_ccdf(special_products_per_company, "num_special_products")

ccdf_powerlaw = fit_power_law(special_products_per_company, "num_special_products")

In [ ]:
# Plot as scatter plot with log scale
sns.scatterplot(x=ccdf_special_products["num_special_products"], y=ccdf_special_products["ccdf"], s=30, edgecolor="none")
sns.scatterplot(x=ccdf_powerlaw["num_special_products"], y=ccdf_powerlaw["ccdf"], s=30, edgecolor="none")
plt.yscale("log")
plt.xscale("log")
plt.xlabel("Number of special products")
plt.ylabel("CCDF")
plt.title("Number of special products per company")

## Number of edges per product

Count how many edges have a product.

In [ ]:
edge_product_df = supply_relations_df.join(
        supply_relation_details_df,
        on="relation_id",
        how="inner",
    )

# Count how many edges have a product (A product can be multiple times on an edge because of multiple car models)
edge_product_count = edge_product_df.group_by("product_name").agg(
    pl.n_unique("relation_id").alias("num_edges"),
).sort("num_edges", descending=True)

edge_product_count

## Source nodes with empty portfolios

Edge products are sampled from the portfolios of source nodes for Monte Carlo simulation. Empty sources need special treatment.

In [ ]:
# Sum products and special products per company
num_products = products_per_company.join(
    special_products_per_company, how="left", on="firm_id", suffix="_special").fill_null(0)
num_products = num_products.with_columns(
    (pl.col("num_products") + pl.col("num_special_products")).alias("total_products"),
)

# Filter firms with no products
firms_with_no_products = num_products.filter(pl.col("total_products") == 0).select("firm_id")
# Filter firms with out-degree > 0
firms_with_out_degree = node_df.filter(pl.col("out_degree") > 0).select("firm_id", "out_degree")

# Join firms with no products and firms with out-degree > 0
firms_with_no_products_and_out_degree = firms_with_no_products.join(
    firms_with_out_degree, how="inner", on="firm_id",
)
total_out_degree = firms_with_no_products_and_out_degree["out_degree"].sum()
print(f"Firms with no products and out-degree > 0: {firms_with_no_products_and_out_degree.height}")
print(f"Total out-degree of firms with no products: {total_out_degree}")
firms_with_no_products_and_out_degree

# Edge-product characteristics

In [ ]:
# Group supply relation details by edge and count
edge_product_df = supply_relations_df.join(supply_relation_details_df, on="relation_id", how="inner")
num_products = edge_product_df.group_by(["source_firm_id", "target_firm_id", "relation_id"]).agg(
    pl.count("relation_id").alias("num_products")).sort("num_products", descending=True)
num_products

In [ ]:
# Get mean and median number of products per edge
mean_num_products = num_products["num_products"].mean()
median_num_products = num_products["num_products"].median()

# Ratio of deterministic edges
ratio = num_products.height / supply_relations_df.height

print(f"Mean number of products per edge: {mean_num_products}")
print(f"Median number of products per edge: {median_num_products}")
print(f"Ratio of deterministic edges: {ratio}")

In [ ]:
kde = gaussian_kde(num_products["num_products"], bw_method="scott")
samples_kde = kde.resample(size=100000)[0]
# Round samples to integers
samples_kde = np.round(samples_kde).astype(int)

In [ ]:
# Create CCDF for plotting
ccdf_products = compute_ccdf(num_products, "num_products")
kde_df = pl.DataFrame({"num_products": samples_kde})
ccdf_kde = compute_ccdf(kde_df, "num_products")

ccdf_powerlaw = fit_power_law(num_products, "num_products")

# Create distribution of number of products per edge
sns.scatterplot(x=ccdf_products["num_products"], y=ccdf_products["ccdf"], s=30, edgecolor="none")
sns.scatterplot(x=ccdf_kde["num_products"], y=ccdf_kde["ccdf"], s=30, edgecolor="none", color="red")
sns.scatterplot(x=ccdf_powerlaw["num_products"], y=ccdf_powerlaw["ccdf"], s=30, edgecolor="none", color="green")
plt.xlabel("Number of products per edge")
plt.ylabel("Frequency")
plt.title("Distribution of number of products per edge")
plt.xscale("log")
plt.yscale("log")
plt.show()

## Correlation between source firm characteristics and number of products

In [ ]:
# Add size info of source firm
firm_info_source = num_products.join(firms_df, how="left", left_on="source_firm_id", right_on="firm_id").select(
    "source_firm_id", "target_firm_id", "num_products", "capital", "sales", "employees")
firm_info_source = firm_info_source.join(node_df, how="left", left_on="source_firm_id", right_on="firm_id")

# Add size info of target firm
firm_info_target = num_products.join(firms_df, how="left", left_on="target_firm_id", right_on="firm_id").select(
    "source_firm_id", "target_firm_id", "num_products", "capital", "sales", "employees")
firm_info_target = firm_info_target.join(node_df, how="left", left_on="target_firm_id", right_on="firm_id")

In [ ]:
# Calculate correlation between num products and capital
numeric_cols = [
    "degree",
    "in_degree",
    "out_degree",
    "closeness_centrality",
    "eigenvector_centrality",
    "pagerank",
    "tier",
    "capital",
    "sales",
    "employees",
]

print("\nCorrelation with source firm characteristics:")

for col in numeric_cols:
    corr = firm_info_source.select(pl.corr("num_products", col)).to_numpy()[0][0]  # Extract single value from 2D array
    print(f"Correlation between num_products and {col}: {corr:.4f}")

print("\nCorrelation with target firm characteristics:")

for col in numeric_cols:
    corr = firm_info_target.select(pl.corr("num_products", col)).to_numpy()[0][0]  # Extract single value from 2D array
    print(f"Correlation between num_products and {col}: {corr:.4f}")

## Correlation between edge centrality and num products

In [ ]:
# Load centrality (computed on server)
with open(paths.ANALYSIS_DIR / "betweenness_centrality.json", "r") as f:
    edge_betweenness_centrality = json.load(f, parse_int=int)

# Convert keys to integer
edge_betweenness_centrality = {int(k): v for k, v in edge_betweenness_centrality.items()}

with open(paths.ANALYSIS_DIR / "edge_load.json", "r") as f:
    edge_load = json.load(f, parse_int=int)

# Convert keys to integer
edge_load = {int(k): v for k, v in edge_load.items()}

edge_df = pl.DataFrame({
    "relation_id": [edge for edge in edge_betweenness_centrality],
    "betweenness_centrality": [edge_betweenness_centrality[edge] for edge in edge_betweenness_centrality],
    "edge_load": [edge_load[edge] for edge in edge_load],
})

In [ ]:
test = edge_df.join(num_products, how="inner", on="relation_id")

# Calculate correlation between num products and capital
numeric_cols = [
    "betweenness_centrality",
    "edge_load",
]

for col in numeric_cols:
    corr = test.select(pl.corr("num_products", col)).to_numpy()[0][0]  # Extract single value from 2D array
    print(f"Correlation between num_products and {col}: {corr:.4f}")

# Correlation analysis

| Correlation Value | Interpretation                  |
|-------------------|--------------------------------|
| 1.0               | Perfect linear relationship     |
| 0.5               | Moderate linear relationship    |
| 0.1 - 0.4         | Weak linear relationship        |
| 0                 | No relationship                 |

## Nodes

In [ ]:
# Join firms with node characteristics
node_df2 = node_df.join(firms_df, how="left", on="firm_id")

# Select numerical columns
numeric_cols = [
    "degree",
    "in_degree",
    "out_degree",
    "closeness_centrality",
    "eigenvector_centrality",
    "pagerank",
    "tier",
    "capital",
    "sales",
    "employees",
]
corr_df = node_df2.select(numeric_cols)
# Filter columns without nulls
cleaned = corr_df.drop_nulls()

# Create correlation matrix
corr_matrix1 = corr_df.corr()
corr_matrix2 = cleaned.corr()

In [ ]:
# Visualize correlation matrix
labels = ["degree", "in_degree", "out_degree", "closeness_centrality", "eigenvector_centrality", "pagerank", "tier"]
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix1.to_pandas(), annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels)

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix2.to_pandas(), annot=True, fmt=".2f", xticklabels=corr_df.columns, yticklabels=corr_df.columns)

## Edges

# Groups

In [ ]:
# Create dict of nodes per cluster
group_node_dict = defaultdict(list)

for node, attr in node_attr.items():
    group_node_dict[attr["group_id"]].append(node)

# List of cluster sizes
group_sizes = [len(group) for group in group_node_dict.values()]

In [ ]:
# Clustering statistics
print("Number of groups:", len(group_sizes))
print("Size of largest group:", max(group_sizes))
print("Size of smallest group:", min(group_sizes))
print("Number of singletons:", sum([1 for size in group_sizes if size == 1]))
print("Average group size:", np.mean(group_sizes))
print("Median group size:", np.median(group_sizes))

# PLot distribution of group sizes
sns.histplot(group_sizes, bins=300, kde=False)
plt.xlabel("Group size")
plt.ylabel("Frequency")
plt.title("Distribution of group sizes")
plt.xlim(0, 25)
plt.ylim(0, 5000)

# Tiers

**How to calculate supply chain tiers**

1. Identify all sinks (out-degree = 0)
2. Perform a breadth-first search from each sink and determine the tier of each successor
3. Calculate the median tier of each node

In [ ]:
# Add num products to node df
node_df = node_df.join(products_per_company, how="left", on="firm_id")

In [ ]:
tier_stats = (
    node_df
    .group_by("tier")
    .agg([
        pl.count("firm_id").alias("num_nodes"),
        pl.mean("degree").alias("mean_degree"),
        pl.mean("out_degree").alias("mean_out_degree"),
        pl.mean("in_degree").alias("mean_in_degree"),
        pl.mean("num_products").alias("mean_products"),
    ])
    .sort("tier")
)

In [ ]:
# Plot nodes by tier
sns.barplot(
    tier_stats,
    x="tier",
    y="num_nodes",
)

In [ ]:
# Plot mean degree per tier with seaborn
sns.barplot(
    tier_stats,
    x="tier",
    y="mean_degree",
)

In [ ]:
# Plot mean in-degree per tier with seaborn
sns.barplot(
    tier_stats,
    x="tier",
    y="mean_in_degree",
)

In [ ]:
# Plot mean out-degree per tier with seaborn
sns.barplot(
    tier_stats,
    x="tier",
    y="mean_out_degree",
)

In [ ]:
# Plot products per tier with seaborn
sns.barplot(
    tier_stats,
    x="tier",
    y="mean_products",
)

# Clustering coefficient

**Definition:** "For directed graphs, the clustering is similarly defined as the fraction of all possible directed triangles or geometric average of the subgraph edge weights for unweighted and weighted directed graph respectively"
- degree 0/1 nodes are automatically excluded because the clustering coefficient is not defined

- Average clustering coefficient of the network is very low which indicates a sparse network.
- the network lacks the high clustering that normally accompanies small-world structure
- Sparse local cohesion – a node’s neighbours almost never know one another, so information tends to spread along paths rather than within tightly knit cliques
- failures or removals of single edges can more easily disconnect parts of the graph, unless compensated by high degree. They propagate linearly through the supply chain because there are few redundant loops. A single critical supplier can sever many paths.
- High-risk hubs (high degree and low neighbourhood connectivity) deserve priority: if they fail, their customers share no alternative supplier in common.
- Tiered / hierarchical flow – Most links run “upstream → downstream.” Once a component leaves a supplier, it rarely comes back from a peer, so triangles are scarce.
- Competitive secrecy – Suppliers serving the same OEM often avoid direct ties to protect know-how.
- Bipartite tendency – Many datasets capture buyer–supplier edges only, not supplier–supplier collaboration, so potential triangles are literally invisible.
- Regulatory constraints – Antitrust rules can discourage horizontal cooperation that would close triangles.

In [ ]:
orig_Cavg = nx.average_clustering(supply_net)
print("The average clustering coefficient is:", orig_Cavg)

Random baseline test
- we create n degree-sequence preserving random graphs and calculate their average clustering coefficients.
- If z ≈ 0 the observed clustering is fully explained by the degree sequence: firms have so few same-tier connections that triangles are just statistical accidents.

In [ ]:
# Random baseline test
n_iter = 100
Cavg_rand = []

for i in range(n_iter):
    # Create a random graph with the same degree sequence
    random_graph = nx.directed_configuration_model(node_in_degree.values(), node_out_degree.values(), create_using=nx.DiGraph)
    Cavg_rand.append(nx.average_clustering(random_graph))

In [ ]:
rand_mean = np.mean(Cavg_rand)
rand_std  = np.std(Cavg_rand)

z = (orig_Cavg - rand_mean) / rand_std
print(f"Original C_avg  : {orig_Cavg: .4f}")
print(f"Random  mean    : {rand_mean: .4f}  ± {rand_std: .4f}")
print(f"z-score         : {z: .2f}") # Times above/below the random mean
print(f"Triangles occur {100 * (orig_Cavg / rand_mean - 1): .2f} % more often than by chance")

In [ ]:
# If your supply graph is directed, ignore direction for now
UG = supply_net.to_undirected()

tri_per_node = nx.triangles(UG)      # {node: #triangles that include it}
# keep only the nodes that actually form at least one triangle
tri_nodes = {n: t for n, t in tri_per_node.items() if t > 0}

# Top-20 triangle hubs
top20 = sorted(tri_nodes.items(), key=lambda x: x[1], reverse=True)[:20]
for n, t in top20:
    print(f"{n:<20}  triangles: {t}")

# Cycles

The low difference between the number of nodes and the number of strongly connected components indicates a small number of cycles. The network has a tiered structure.

A strongly connected component is a maximal set of nodes (vertices) such that, for every pair of nodes u and v in that set, there is a directed path from u to v and also a directed path from v back to u.

In [ ]:
# Get number of strongly connected components
num_components = nx.number_strongly_connected_components(supply_net)

# Count number of nodes
num_nodes = len(supply_net.nodes)

num_nodes, num_components

# Assortativity

- degree assortativity in the network is nearly neutral
- Your supply network is almost assortatively neutral: it doesn’t strongly prefer “like-with-like” or “hub-to-spoke” pairings.  The slight disassortativity you do observe is typical but so modest that it’s unlikely to drive systemic risk by itself. Keep the emphasis on identifying critical hubs and poorly connected dependents rather than on network-wide mixing patterns.

In [ ]:
# G is your directed supply-chain graph, with edge direction
#   supplier  ──▶  customer
# (so a supplier’s customers count toward its OUT-degree)

r_out_in  = nx.degree_pearson_correlation_coefficient(supply_net, x='out', y='in')
r_out_out = nx.degree_pearson_correlation_coefficient(supply_net, x='out', y='out')
r_in_out  = nx.degree_pearson_correlation_coefficient(supply_net, x='in',  y='out')
r_in_in   = nx.degree_pearson_correlation_coefficient(supply_net, x='in',  y='in')

print(f"r_out_in  (supplier-out  ➜  buyer-in)  = {r_out_in : .3f}")
print(f"r_out_out (supplier-out  ➜  buyer-out) = {r_out_out: .3f}")
print(f"r_in_out  (supplier-in   ➜  buyer-out) = {r_in_out : .3f}")
print(f"r_in_in   (supplier-in   ➜  buyer-in)  = {r_in_in  : .3f}")

# Average path length

Computing the exact average shortest path length on the undirected projection of the LCC ($N \approx 32{,}668$ nodes) requires $\mathcal{O}(N^2)$ BFS operations and is computationally expensive. We approximate it via uniform sampling of $K$ source nodes and aggregating the resulting $K \cdot N$ path lengths (excluding self-loops). This is standard practice for large-network analysis and gives confidence intervals tight enough for small-world quantification.

In combination with the clustering coefficient from the previous section, we compute the Watts–Strogatz small-world index

$$\sigma = \frac{C / C_{\text{random}}}{L / L_{\text{random}}}$$

where $L_{\text{random}} = \log N / \log \langle k \rangle$ is the expected path length of a random graph with the same size and mean degree. $\sigma > 1$ indicates small-world structure.

In [ ]:
# Average shortest path length via uniform source sampling
import random
import time

# Use the undirected projection of the LCC built in the bow-tie / community sections
lcc_undir_apl = supply_net.subgraph(
    max(nx.weakly_connected_components(supply_net), key=len)
).to_undirected()
N_lcc = lcc_undir_apl.number_of_nodes()
E_lcc = lcc_undir_apl.number_of_edges()

# Uniform source sample
random.seed(42)
K_sources = 1000
sample_sources = random.sample(list(lcc_undir_apl.nodes()), K_sources)

# Collect path lengths (BFS from each source)
_t = time.time()
path_lengths = []
for i, src in enumerate(sample_sources):
    lengths = nx.single_source_shortest_path_length(lcc_undir_apl, src)
    path_lengths.extend(L for L in lengths.values() if L > 0)
    if (i + 1) % 250 == 0:
        print(f"  {i+1}/{K_sources} sources processed ({time.time()-_t:.1f}s)")

mean_apl   = np.mean(path_lengths)
median_apl = np.median(path_lengths)
std_apl    = np.std(path_lengths)

print()
print(f"LCC (undirected):              {N_lcc:,} nodes, {E_lcc:,} edges")
print(f"Sample size K:                 {K_sources} sources")
print(f"Total path-length samples:     {len(path_lengths):,}")
print(f"Mean shortest path length:     {mean_apl:.3f}")
print(f"Median shortest path length:   {median_apl:.1f}")
print(f"Standard deviation:            {std_apl:.3f}")
print(f"(Diameter on undirected LCC:   11)")

In [ ]:
# Watts-Strogatz small-world index sigma
mean_degree = 2 * E_lcc / N_lcc
L_random = np.log(N_lcc) / np.log(mean_degree)

# Clustering coefficients from the earlier section (cells ~55 / ~58)
C_observed = 0.0118
C_random   = 0.0082

sigma_small_world = (C_observed / C_random) / (mean_apl / L_random)

print(f"Mean degree <k>:               {mean_degree:.2f}")
print(f"L (observed, sampled):         {mean_apl:.3f}")
print(f"L_random = log(N)/log(<k>):    {L_random:.3f}")
print(f"C (observed):                  {C_observed}")
print(f"C_random (deg-seq baseline):   {C_random}")
print()
print(f"  C / C_random = {C_observed/C_random:.3f}")
print(f"  L / L_random = {mean_apl/L_random:.3f}")
print(f"  sigma        = {sigma_small_world:.3f}    (> 1 → small-world)")

In [ ]:
nx.core_number(supply_net)

# Countries
- Which countries are at the beginning of the supply chain?
- Which products or clusters are at the beginning of the supply chain?
- Which countries are connected?
- Top products by country

In [ ]:
countries_df = (
    node_df
    .filter(pl.col("nation_name").is_not_null())
    .group_by("nation_name")
    .agg([
        pl.count("firm_id").alias("num_firms"),
        pl.mean("degree").alias("mean_degree"),
        pl.mean("in_degree").alias("mean_in_degree"),
        pl.mean("out_degree").alias("mean_out_degree"),
    ])
    # e.g. sort by num_firms (or pick whichever column makes sense)
    .sort("num_firms", descending=True)
)

In [ ]:
# Plot degrees by country
fig, ax = plt.subplots(figsize=(16, 6))
country_degree_plot = sns.barplot(
    countries_df,
    y="num_firms",
    x="nation_name",
    ax=ax,
)
country_degree_plot.set(
    xlabel="Number of firms",
    ylabel="Country",
    title="Number of firms by country",
    )
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# PLot mean degree by country
fig, ax = plt.subplots(figsize=(16, 6))
country_mean_degree_plot = sns.barplot(
    countries_df.sort("mean_degree", descending=True),
    y="mean_degree",
    x="nation_name",
    ax=ax,
)
country_mean_degree_plot.set(
    xlabel="Mean degree",
    ylabel="Country",
    title="Mean degree by country",
    )
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Convert your Polars DataFrame to pandas
# -----------------------------------------------
# (Assuming `countries_df` is a Polars DataFrame with columns
#  ["nation_name", "num_firms", "mean_degree", "mean_in_degree", "mean_out_degree"])
df = countries_df.to_pandas()

# Step 2: Melt (unpivot) so that each row is one (country, metric, value)
# ------------------------------------------------------------------------
df_long = df.melt(
    id_vars="nation_name",
    value_vars=["mean_in_degree", "mean_out_degree"],
    var_name="metric",
    value_name="value",
)

# Now df_long will look like:
#    nation_name         metric  value
# 0         USA    mean_degree   3.14
# 1       China    mean_degree   2.87
# 2     Germany    mean_degree   3.05
# ...        ...            ...    ...
#  (then again for mean_in_degree, then for mean_out_degree)

# Step 3: Sort countries by one of the metrics, if desired
# ---------------------------------------------------------
# If you want the countries ordered by, say, overall mean_degree descending:
order = (
    df.sort_values("mean_degree", ascending=False)
      .nation_name
      .tolist()
)

# Step 4: Plot with Seaborn, using hue="metric"
# ---------------------------------------------
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(16, 6))

sns.barplot(
    data=df_long,
    x="nation_name",
    y="value",
    hue="metric",
    order=order,            # ensures consistent ordering of countries
    palette="Set2",         # any qualitative palette works (e.g. Set2, Paired, etc.)
    ax=ax,
)

ax.set_xlabel("Country", fontsize=12)
ax.set_ylabel("Value", fontsize=12)
ax.set_title("Mean Degree / In‐Degree / Out‐Degree by Country", fontsize=14)

plt.xticks(rotation=90)
plt.legend(title="Metric", loc="upper right")
plt.tight_layout()
plt.show()

# Community analysis

We detect communities on the **undirected projection of the LCC** using the Louvain algorithm with a fixed random seed for reproducibility. Newman modularity $Q$ is computed for the resulting partition and benchmarked against two structural baselines:

1. A **country-based partition** (one community per `nation_id`) — does geography alone explain the modular structure?
2. A **tier-based partition** (one community per median BFS-tier) — does the supply-chain hierarchy explain it?

A higher $Q$ indicates a more modular partition. Empirical supply networks typically report $Q \in [0.4, 0.7]$ for Louvain partitions.

In [ ]:
# Undirected LCC subgraph for community detection
lcc_nodes = max(nx.weakly_connected_components(supply_net), key=len)
lcc_undir = supply_net.subgraph(lcc_nodes).to_undirected()

# Louvain partition (deterministic seed for reproducibility)
louvain_partition = nx.community.louvain_communities(lcc_undir, seed=42)
Q_louvain = nx.community.modularity(lcc_undir, louvain_partition)

community_sizes = sorted([len(c) for c in louvain_partition], reverse=True)
print(f"LCC nodes:                        {lcc_undir.number_of_nodes()}")
print(f"LCC edges (undirected):           {lcc_undir.number_of_edges()}")
print(f"Number of Louvain communities:    {len(louvain_partition)}")
print(f"Newman modularity Q (Louvain):    {Q_louvain:.4f}")
print(f"Largest community size:           {community_sizes[0]}")
print(f"Median community size:            {int(np.median(community_sizes))}")
print(f"Communities with > 100 firms:     {sum(1 for s in community_sizes if s > 100)}")
print(f"Communities with > 1000 firms:    {sum(1 for s in community_sizes if s > 1000)}")
print(f"Top-10 community sizes:           {community_sizes[:10]}")

In [ ]:
# Baseline 1: country-based partition modularity
country_partition_dict = defaultdict(set)
for n in lcc_undir.nodes():
    nat = supply_net.nodes[n].get("nation_id")
    country_partition_dict[nat].add(n)
country_partition = list(country_partition_dict.values())
Q_country = nx.community.modularity(lcc_undir, country_partition)

# Baseline 2: tier-based partition modularity
tier_partition_dict = defaultdict(set)
for n in lcc_undir.nodes():
    t = node_tier_dict.get(n, -1)
    tier_partition_dict[t].add(n)
tier_partition = list(tier_partition_dict.values())
Q_tier = nx.community.modularity(lcc_undir, tier_partition)

print(f"Number of country groups on LCC:   {len(country_partition)}")
print(f"Number of tier groups on LCC:      {len(tier_partition)}")
print()
print(f"Modularity Q (country baseline):   {Q_country:.4f}")
print(f"Modularity Q (tier baseline):      {Q_tier:.4f}")
print(f"Modularity Q (Louvain):            {Q_louvain:.4f}")

In [ ]:
# Community size distribution (log-log)
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(community_sizes, bins=40, log_scale=(True, True), ax=ax)
ax.set_xlabel("Community size (number of firms)")
ax.set_ylabel("Number of communities")
ax.set_title(f"Louvain community size distribution ({len(louvain_partition)} communities, Q={Q_louvain:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# Country composition of the top-k largest communities
top_k = 10
top_communities = sorted(louvain_partition, key=len, reverse=True)[:top_k]

# Build node_id -> nation_name lookup from node_df (country-joined in cell 5)
nation_name_lookup = dict(zip(node_df["firm_id"], node_df["nation_name"]))

print(f"Top-{top_k} largest Louvain communities and their dominant countries:")
print(f"{'Rank':<5} {'Size':<8} {'Top-3 countries (share)':<60}")
print("-" * 75)
for rank, comm in enumerate(top_communities, 1):
    country_counts = Counter()
    for n in comm:
        nat_name = nation_name_lookup.get(n) or "Unknown"
        country_counts[nat_name] += 1
    top3 = country_counts.most_common(3)
    top3_str = ", ".join(f"{c}={cnt/len(comm)*100:.0f}%" for c, cnt in top3)
    print(f"{rank:<5} {len(comm):<8} {top3_str:<60}")

# Diameter

The diameter is necessary to determine the number of iterations the Zhao simulation model should run. The diameter of the Marklines network is 11. The computation takes around 30 minutes.

In [ ]:
cc = nx.weakly_connected_components(supply_net)
largest_cc = max(cc, key=len)
largest_cc_subgraph = supply_net.subgraph(largest_cc).copy()
print(largest_cc_subgraph.number_of_nodes())
print(f"Diameter: {nx.diameter(largest_cc_subgraph.to_undirected())}")

# Bow-tie structure

Following Broder et al. (2000)'s decomposition for directed networks, we partition the supply graph into the canonical bow-tie components:

- **LSCC** — Largest Strongly Connected Component (mutual reachability)
- **IN** — nodes that reach the LSCC but cannot be reached from it (upstream)
- **OUT** — nodes reachable from the LSCC but cannot reach it (downstream)
- **TUBES** — nodes reachable from IN *and* can reach OUT, but not in LSCC/IN/OUT (alternative paths bypassing the SCC)
- **TENDRILS_IN** — reachable from IN but cannot reach OUT (dead-end downstream of IN)
- **TENDRILS_OUT** — can reach OUT but not reachable from IN (upstream that bypasses LSCC/IN)
- **DISCONNECTED** — none of the above (isolated firms, separate WCCs, and nodes only weakly connected to the LSCC)

For web graphs and dense complex networks the LSCC typically dominates the WCC. For supply networks that approximate a DAG, the LSCC is small or near-trivial, and the bow-tie statistics characterise how upstream supplier hierarchies channel into downstream customer/OEM components.

In [ ]:
# SCC-size distribution as a precursor to the bow-tie decomposition
import time
_t = time.time()
sccs_list = sorted(nx.strongly_connected_components(supply_net), key=len, reverse=True)
multi_sccs = [s for s in sccs_list if len(s) > 1]
print(f"Number of SCCs:                   {len(sccs_list):,}")
print(f"Number of multi-node SCCs:        {len(multi_sccs)}")
print(f"Top-10 SCC sizes:                 {[len(s) for s in sccs_list[:10]]}")
print(f"LSCC size:                        {len(sccs_list[0])} "
      f"({len(sccs_list[0])/supply_net.number_of_nodes()*100:.3f}% of all nodes)")
print(f"(computed in {time.time()-_t:.1f}s)")

In [ ]:
# Bow-tie decomposition (Broder et al. 2000)
_t = time.time()
lscc_set = sccs_list[0]
lscc_node = next(iter(lscc_set))

# Ancestors and descendants of LSCC give IN and OUT
ancestors = nx.ancestors(supply_net, lscc_node)
descendants = nx.descendants(supply_net, lscc_node)
in_comp = ancestors - lscc_set
out_comp = descendants - lscc_set

# Forward reachability from IN via virtual source
g_aug = supply_net.copy()
src_virtual = "__BOWTIE_SRC__"
g_aug.add_edges_from((src_virtual, n) for n in in_comp)
reachable_from_in = nx.descendants(g_aug, src_virtual)
g_aug.remove_node(src_virtual)

# Backward reachability to OUT via virtual sink
tgt_virtual = "__BOWTIE_TGT__"
g_aug.add_edges_from((n, tgt_virtual) for n in out_comp)
can_reach_out = nx.ancestors(g_aug, tgt_virtual)
g_aug.remove_node(tgt_virtual)

# Mutually exclusive classification
classified_set = lscc_set | in_comp | out_comp
tubes = (reachable_from_in & can_reach_out) - classified_set
classified_set = classified_set | tubes
tendrils_in = reachable_from_in - classified_set
classified_set = classified_set | tendrils_in
tendrils_out = can_reach_out - classified_set
classified_set = classified_set | tendrils_out
disconnected = set(supply_net.nodes()) - classified_set

total = supply_net.number_of_nodes()
bowtie_components = {
    "LSCC":         lscc_set,
    "IN":           in_comp,
    "OUT":          out_comp,
    "TUBES":        tubes,
    "TENDRILS_IN":  tendrils_in,
    "TENDRILS_OUT": tendrils_out,
    "DISCONNECTED": disconnected,
}

print(f"{'Component':<14} {'Size':>10}  {'%':>7}")
print("-" * 35)
for name, comp in bowtie_components.items():
    print(f"{name:<14} {len(comp):>10,}  {len(comp)/total*100:>6.2f}%")
print("-" * 35)
print(f"{'TOTAL':<14} {total:>10,}  100.00%")
print(f"(decomposition computed in {time.time()-_t:.1f}s)")

In [ ]:
# Pie chart of the bow-tie partition
labels = list(bowtie_components.keys())
sizes  = [len(bowtie_components[k]) for k in labels]
colors = ["#d62728", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct=lambda p: f"{p:.1f}%\n({int(round(p*sum(sizes)/100)):,})",
    startangle=90,
    pctdistance=0.78,
    textprops={"fontsize": 10},
)
ax.set_title(
    f"Bow-tie decomposition of the MarkLines supply network\n"
    f"(LSCC = {len(bowtie_components['LSCC'])} nodes ≈ "
    f"{len(bowtie_components['LSCC'])/total*100:.2f}% of {total:,})"
)
plt.tight_layout()
plt.show()

# Nexus suppliers

- not first-tier suppliers
- Operational nexus supplier: high Eigenvector centrality, high degree centrality, high betweenness centrality